In [1]:
import numpy as np
import pandas as pd

In [3]:
file_path = r'C:\Users\leeji\Desktop\study_csv\Technology.csv'

try:
  # 한글 인코딩(utf-8 또는 cp949) 자동 대응
  try:
    df = pd.read_csv(file_path, encoding='utf-8')
  except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')
  print(f"'{file_path}' 인코딩 성공")
except FileNotFoundError:
  print(f"⚠️ '{file_path}' 인코딩 실패")

'C:\Users\leeji\Desktop\study_csv\Technology.csv' 인코딩 성공


In [4]:
df.columns = df.columns.str.strip()

In [5]:
num_cols = [
    '2027 사전예고',
    '2026 사전 예고',
    '2026 최종 일반',
    '2025 사전예고',
    '2025 최종일반',
    '2024 사전예고',
    '2024 최종일반',
    '2023 사전예고',
    '2023 최종일반',
]
for col in num_cols:
  if col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

In [6]:
df['가티오_증감인원(26-27)'] = df['2027 사전예고'] - df['2026 사전 예고']
df['가티오_증감률(%)(26-27)'] = np.where(
    df['2026 사전 예고'] != 0,
    ((df['2027 사전예고'] - df['2026 사전 예고']) / df['2026 사전 예고']) * 100,
    0,
).round(2)

In [7]:
df['2026_배수'] = np.where(
    df['2026 사전 예고'] != 0, df['2026 최종 일반'] / df['2026 사전 예고'], 1.0
)
df['2025_배수'] = np.where(
    df['2025 사전예고'] != 0, df['2025 최종일반'] / df['2025 사전예고'], 1.0
)
df['2024_배수'] = np.where(
    df['2024 사전예고'] != 0, df['2024 최종일반'] / df['2024 사전예고'], 1.0
)

# 3개년 평균 배수 (단년도 변동성 완화)
df['최근3개년_평균배수'] = (
    (df['2026_배수'] + df['2025_배수'] + df['2024_배수']) / 3
).round(2)

In [8]:
df['2027_예측최종(3개년평균배수)'] = (
    (df['2027 사전예고'] * df['최근3개년_평균배수']).round().astype(int)
)

In [9]:
total_row = pd.DataFrame([{
    '지역': '합계',
    '2026 사전 예고': df['2026 사전 예고'].sum(),
    '2026 최종 일반': df['2026 최종 일반'].sum(),
    '2027 사전예고': df['2027 사전예고'].sum(),
    '가티오_증감인원(26-27)': df['2027 사전예고'].sum()
    - df['2026 사전 예고'].sum(),
    '가티오_증감률(%)(26-27)': (
        round(
            (
                (df['2027 사전예고'].sum() - df['2026 사전 예고'].sum())
                / df['2026 사전 예고'].sum()
            )
            * 100,
            2,
        )
        if df['2026 사전 예고'].sum() != 0
        else 0
    ),
    '최근3개년_평균배수': round(df['최근3개년_평균배수'].mean(), 2),
    '2027_예측최종(3개년평균배수)': df['2027_예측최종(3개년평균배수)'].sum(),
}])

df_result = pd.concat([df, total_row], ignore_index=True)

# 주요 예측 결과 출력
output_cols = [
    '지역',
    '2026 사전 예고',
    '2026 최종 일반',
    '2027 사전예고',
    '가티오_증감인원(26-27)',
    '가티오_증감률(%)(26-27)',
    '최근3개년_평균배수',
    '2027_예측최종(3개년평균배수)',
]
display(df_result[output_cols])

,지역,2026 사전 예고,2026 최종 일반,2027 사전예고,가티오_증감인원(26-27),가티오_증감률(%)(26-27),최근3개년_평균배수,2027_예측최종(3개년평균배수)
0,서울,19,20,15,-4,-21.05,0.97,15
1,경기,38,56,31,-7,-18.42,1.31,41
2,인천,16,20,8,-8,-50.00,1.16,9
3,대전,1,2,0,-1,-100.00,1.33,0
4,대구,2,5,5,3,150.00,1.50,8
5,울산,2,3,2,0,0.00,1.17,2
6,부산,7,7,2,-5,-71.43,0.93,2
7,광주,1,1,0,-1,-100.00,1.00,0
8,세종,0,2,2,2,0.00,0.89,2
9,강원,0,0,0,0,0.00,1.17,0
